In [1]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader

In [2]:
os.makedirs("models", exist_ok=True)
os.makedirs("results", exist_ok=True)

In [3]:
with open("records_long.json", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)

In [4]:
df = df[["text", "model"]].copy()
df = df.dropna(subset=["text", "model"])
df["text"] = df["text"].astype(str)

In [5]:

label_mapping = {
    "chatgpt": "OpenAI",
    "gemma": "Google",
    "human": "Human",
    "llama": "Meta",
    "mistral": "Anthropic"
}

df["final_label"] = df["model"].map(label_mapping)

print("\nMapped training labels:")
print(df["final_label"].value_counts())

# Remove rows that were not mapped, if any
df = df.dropna(subset=["final_label"])

print("\nShape after label mapping:", df.shape)


Mapped training labels:
final_label
Meta         86120
Anthropic    81448
Google       69388
Human        49050
OpenAI       34587
Name: count, dtype: int64

Shape after label mapping: (320593, 3)


In [6]:
df = (
    df.groupby("model", group_keys=False)
      .apply(lambda x: x.sample(frac=0.5, random_state=42))
      .reset_index(drop=True)
)

C:\Users\franc\AppData\Local\Temp\ipykernel_10660\580186971.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(frac=0.5, random_state=42))


In [7]:
df = (
    df.groupby("model", group_keys=False)
      .apply(lambda x: x.sample(frac=0.5, random_state=42))
      .reset_index(drop=True)
)

C:\Users\franc\AppData\Local\Temp\ipykernel_10660\580186971.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(frac=0.5, random_state=42))


In [8]:
encoder = LabelEncoder()
y = encoder.fit_transform(df["final_label"])
texts = df["text"]

print("\nFinal class mapping:")
for i, class_name in enumerate(encoder.classes_):
    print(f"{i} -> {class_name}")




Final class mapping:
0 -> Anthropic
1 -> Google
2 -> Human
3 -> Meta
4 -> OpenAI


In [9]:
X_train_texts, X_val_texts, y_train, y_val = train_test_split(
    texts,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTraining samples:", len(X_train_texts))
print("Validation samples:", len(X_val_texts))



Training samples: 64118
Validation samples: 16030


In [10]:
vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words="english",
    dtype=np.float32
)

X_train = vectorizer.fit_transform(X_train_texts)
X_val = vectorizer.transform(X_val_texts)



In [11]:
class SparseTextDataset(Dataset):
    def __init__(self, X_sparse, y):
        self.X = X_sparse
        self.y = y

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        x = self.X[idx].toarray().squeeze().astype(np.float32)
        y = self.y[idx]
        return torch.from_numpy(x), torch.tensor(y, dtype=torch.long)


train_dataset = SparseTextDataset(X_train, y_train)
val_dataset = SparseTextDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)


In [12]:
class DNNClassifier(nn.Module):
    def __init__(self, input_size, num_classes):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.network(x)


input_size = X_train.shape[1]
num_classes = len(encoder.classes_)

model = DNNClassifier(input_size, num_classes)

print("\nModel created successfully:")
print(model)



Model created successfully:
DNNClassifier(
  (network): Sequential(
    (0): Linear(in_features=5000, out_features=256, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=256, out_features=128, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=128, out_features=5, bias=True)
  )
)


In [13]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


In [14]:
epochs = 5

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()

        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.4f}")

Epoch 1/5 - Loss: 0.6866
Epoch 2/5 - Loss: 0.4212
Epoch 3/5 - Loss: 0.3331
Epoch 4/5 - Loss: 0.2447
Epoch 5/5 - Loss: 0.1499


In [15]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for X_batch, y_batch in val_loader:
        outputs = model(X_batch)
        _, predicted = torch.max(outputs, 1)

        total += y_batch.size(0)
        correct += (predicted == y_batch).sum().item()

accuracy = correct / total
print(f"\nValidation accuracy: {accuracy:.4f}")


Validation accuracy: 0.8001


In [16]:
torch.save(model.state_dict(), "models/tfidf_dnn_model.pth")
joblib.dump(vectorizer, "models/tfidf_vectorizer.pkl")
joblib.dump(encoder, "models/label_encoder.pkl")

['models/label_encoder.pkl']

In [23]:
df_subm = pd.read_csv("subm1.csv",sep=";")
print("\nSubmission file loaded.")
print(df_subm.head())
print("\nSubmission columns:", df_subm.columns.tolist())
print("Submission shape:", df_subm.shape)


Submission file loaded.
     ID                                               Text
0  D2-1  A covalent bond is a chemical bond that involv...
1  D2-2  A covalent bond forms when two atoms share one...
2  D2-3  A covalent bond is a type of chemical bond whe...
3  D2-4  A covalent bond is a chemical bond that involv...
4  D2-5  Driven by exciting developments in the field o...

Submission columns: ['ID', 'Text']
Submission shape: (150, 2)


In [24]:
possible_text_columns = ["text", "Text", "sentence", "Sentence", "content", "Content"]

text_column = None
for col in possible_text_columns:
    if col in df_subm.columns:
        text_column = col
        break

if text_column is None:
    raise ValueError(
        "Could not find a text column in subm1.csv. "
        "Rename the correct column or edit 'possible_text_columns'."
    )

print(f"\nUsing text column for prediction: {text_column}")

df_subm[text_column] = df_subm[text_column].astype(str)


Using text column for prediction: Text


In [25]:
X_subm = vectorizer.transform(df_subm[text_column]).toarray()

In [26]:
predictions = []

model.eval()
with torch.no_grad():
    for start in range(0, X_subm.shape[0], 64):
        end = start + 64

        batch = X_subm[start:end].astype(np.float32)
        batch_tensor = torch.from_numpy(batch)

        outputs = model(batch_tensor)
        _, predicted = torch.max(outputs, 1)

        predictions.extend(predicted.cpu().numpy())

predicted_labels = encoder.inverse_transform(predictions)




In [28]:
df_final = df_subm[["ID",text_column]].copy()
df_final["Label"] = predicted_labels

os.makedirs("Subm1", exist_ok=True)

output_path = "Subm1/subm1-g5-MEI-B.csv"
df_final.to_csv(output_path, index=False)

print("\nSubmission file saved to:", output_path)
print("\nSubmission preview:")
print(df_final.head(20))
print("\nSubmission shape:", df_final.shape)


Submission file saved to: Subm1/subm1-g5-MEI-B.csv

Submission preview:
       ID                                               Text   Label
0    D2-1  A covalent bond is a chemical bond that involv...  OpenAI
1    D2-2  A covalent bond forms when two atoms share one...   Human
2    D2-3  A covalent bond is a type of chemical bond whe...   Human
3    D2-4  A covalent bond is a chemical bond that involv...  OpenAI
4    D2-5  Driven by exciting developments in the field o...  OpenAI
5    D2-6  Ionic bonding results from the electrostatic a...   Human
6    D2-7  Plate tectonics is the scientific theory expla...   Human
7    D2-8  Plate tectonics is a fundamental scientific th...   Human
8    D2-9  Tectonic plates are relatively rigid and float...   Human
9   D2-10  At present, young oceanic lithosphere is posit...   Human
10  D2-11  Plate tectonics is the scientific theory that ...  Google
11  D2-12  Developed from the 1950s through the 1970s, pl...    Meta
12  D2-13  Introductory textbo